In [ ]:
import sys
from pathlib import Path
_ROOT = Path.cwd().parent.resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from    pathlib         import  Path
from    tqdm.notebook   import  tqdm, trange
from    time            import  perf_counter

import  torch
from    torch.utils.data    import  TensorDataset, DataLoader
import  matplotlib.pyplot   as  plt

from    core.utils    import  velocity_grid, relative_error
from    core.numerical.math   import  area_of_unit_sphere
from    core.numerical.solver     import  LAMBDA, FastSM_Boltzmann_VHS
from    core.numerical.distribution   import  maxwellian_homogeneous, compute_moments_homogeneous


from    loader  import  load_dataset


DEVICE = torch.device('cuda:0')
base_path = _ROOT
assert base_path.exists()

In [ ]:
DIMENSION   = 3
RESOLUTION  = 2**6
MAX_V       = 3.0/LAMBDA

COL_COEFF   = 1/(0.5*torch.pi)
EXP_SPEED   = 0.0

BATCH_SIZE  = 1

In [ ]:
v_grid = velocity_grid(DIMENSION, RESOLUTION, MAX_V)
f_init = maxwellian_homogeneous(v_grid, torch.ones(1, 1), torch.zeros(1, DIMENSION), torch.ones(1, 1))
f_init = f_init.reshape(1, *v_grid.shape[:-1], 1)
param = torch.zeros(1, 1)

# Generate dataloader
test_dataloader = DataLoader(TensorDataset(f_init, param), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
fsm = FastSM_Boltzmann_VHS(
    dimension   = DIMENSION,
    v_num_grid  = RESOLUTION,
    v_max       = MAX_V,
    vhs_coeff   = COL_COEFF,
    vhs_alpha   = EXP_SPEED,
    dtype       = torch.get_default_dtype(),
).to(DEVICE)
v_grid = velocity_grid(DIMENSION, RESOLUTION, MAX_V, device=DEVICE)

In [ ]:
torch.cuda.reset_peak_memory_stats(DEVICE)
torch.cuda.synchronize(DEVICE)

elapse_time = perf_counter()
with torch.inference_mode():
    for f_init, params in test_dataloader:
        f_pred = f_init.to(DEVICE)
        params = params.to(DEVICE)
        f_pred_fft: torch.Tensor = torch.fft.fftn(f_pred, dim=tuple(range(-1-DIMENSION, -1)), norm='forward')
        for _ in range(DIMENSION):  f_pred_fft.unsqueeze_(1)
        q_pred_fft = fsm.compute_fft(None, f_pred_fft)
        q_pred = torch.real(torch.fft.ifftn(q_pred_fft, dim=tuple(range(-1-DIMENSION, -1)), norm='forward'))
        for _ in range(DIMENSION):  q_pred.squeeze_(1)
elapse_time = perf_counter() - elapse_time

torch.cuda.synchronize(DEVICE)
peak_memory_fno = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)
print(f"Elapsed time: {elapse_time:.4e} seconds")
print(f"Peak memory usage: {peak_memory_fno:.2f} MB")

In [ ]:
%%timeit -r 30 -n 5
with torch.inference_mode():
    for f_init, params in test_dataloader:
        f_pred = f_init.to(DEVICE)
        params = params.to(DEVICE)
        f_pred_fft: torch.Tensor = torch.fft.fftn(f_pred, dim=tuple(range(-1-DIMENSION, -1)), norm='forward')
        for _ in range(DIMENSION):  f_pred_fft.unsqueeze_(1)
        q_pred_fft = fsm.compute_fft(None, f_pred_fft)
        q_pred = torch.real(torch.fft.ifftn(q_pred_fft, dim=tuple(range(-1-DIMENSION, -1)), norm='forward'))
        for _ in range(DIMENSION):  q_pred.squeeze_(1)

End of file